# Mars · 03 — Pair features (5 dimensionless features)

**Primary interface** for the pair-feature step. This notebook calls the shared
`channel_heads.features` package (`line_direction_first_n_meters`,
`sample_path_coords_along_line`, `compute_azimuth`, `azimuth_difference`,
`compute_proximity_profile`) — the same functions the batch wrapper
`scripts/build_mars_pair_features_5feat.py` uses.

It is **read-only**: it reads the already-built feature table and pair
geometries, summarises the five model features, and *reproduces* two of them for
one pair straight from the package — proving the math lives in the package, not
in the notebook. The full feature tables are written by the batch wrapper.

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd

from channel_heads.io.paths import PROJECT_ROOT
from channel_heads.features import (
    DIRECTION_SAMPLE_DISTANCE_M,
    N_PROXIMITY_SAMPLES,
    azimuth_difference,
    compute_azimuth,
    compute_proximity_profile,
    line_direction_first_n_meters,
    sample_path_coords_along_line,
)

READY_PARQUET = PROJECT_ROOT / "data/Mars/model_inputs/mars_pair_features_5feat_model_ready.parquet"
PAIRS_GPKG = PROJECT_ROOT / "data/Mars/topology/mars_vn_pairs.gpkg"

MODEL_FEATURES = [
    "orientation_diff_deg",
    "headhead_dist_norm",
    "apex_angle_deg",
    "strahler_order_diff",
    "proximity_profile_norm",
]
print("features table exists:", READY_PARQUET.exists())

features table exists: True


## The five model features (read-only)

In [2]:
feats = pd.read_parquet(READY_PARQUET)
print(f"{len(feats)} model-ready pairs")
feats[MODEL_FEATURES].describe().T

3785 model-ready pairs


,count,mean,std,min,25%,50%,75%,max
orientation_diff_deg,3785.0,35.271521,30.303334,0.000000,11.936497,27.349888,49.279206,180.000000
headhead_dist_norm,3785.0,0.306183,0.167123,0.000000,0.174853,0.281688,0.408902,0.936319
apex_angle_deg,3785.0,27.985337,21.210367,0.000000,12.087188,23.171950,38.406898,164.490551
strahler_order_diff,3785.0,0.709379,0.698518,0.000000,0.000000,1.000000,1.000000,4.000000
proximity_profile_norm,3781.0,0.667374,0.081816,0.395935,0.604156,0.668028,0.726874,0.896382


## Reproduce two features for one pair, via the package

Take one pair's two branch polylines (head→confluence) and recompute
`orientation_diff_deg` and `proximity_profile_norm` exactly as the wrapper does,
then compare to the stored values.

In [3]:
paths = gpd.read_file(PAIRS_GPKG, layer="mars_pair_paths")
both = paths.groupby("pair_id")["branch"].nunique()
both_ids = set(both[both >= 2].index)

cand = feats[
    feats["pair_id"].isin(both_ids)
    & feats["orientation_diff_deg"].notna()
    & feats["proximity_profile_norm"].notna()
]
row = cand.iloc[0]
pid = row["pair_id"]


def branch_coords(pair_id, branch):
    g = paths[(paths["pair_id"] == pair_id) & (paths["branch"] == branch)].geometry.iloc[0]
    return np.asarray(g.coords)


ca, cb = branch_coords(pid, "A"), branch_coords(pid, "B")

# orientation_diff_deg
va, _ = line_direction_first_n_meters(ca, DIRECTION_SAMPLE_DISTANCE_M)
vb, _ = line_direction_first_n_meters(cb, DIRECTION_SAMPLE_DISTANCE_M)
orient = azimuth_difference(compute_azimuth(*va), compute_azimuth(*vb))

# proximity_profile_norm
sa = sample_path_coords_along_line(ca, N_PROXIMITY_SAMPLES)
sb = sample_path_coords_along_line(cb, N_PROXIMITY_SAMPLES)
_, _, prox = compute_proximity_profile(sa, sb)

print(f"pair {pid}")
print(f"  orientation_diff_deg : recomputed={orient:8.3f}  stored={row['orientation_diff_deg']:8.3f}")
print(f"  proximity_profile_norm: recomputed={prox:8.4f}  stored={row['proximity_profile_norm']:8.4f}")

pair 1_5_3_4
  orientation_diff_deg : recomputed=   6.942  stored=   6.942
  proximity_profile_norm: recomputed=  0.6647  stored=  0.6647


---
Full-dataset run (writes the feature tables all downstream steps consume):

```bash
python scripts/build_mars_pair_features_5feat.py
```